In [46]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv.ipython import load_dotenv

In [47]:
load_dotenv(override=True)

True

In [48]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [49]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant"
)

In [50]:
agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="You are a helpful assistant"
)

In [51]:
resp = agent.invoke(input={"messages":[
    {"role":"user","content":"Je m'appelle Saad"}                                    
]})

In [52]:
print(resp['messages'][-1].content)

Enchanté, Saad ! Comment puis-je vous aider aujourd'hui ?


In [53]:
resp = agent.invoke(input={"messages":[
    {"role":"user","content":"Comment je m'appelle?"}                                    
]})

In [54]:
print(resp['messages'][-1].content)

Je suis désolé, mais je ne connais pas ton prénom. Pourrais-tu me le dire ?


In [55]:
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [56]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
advanced_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [57]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest, handler)->ModelResponse:
    env = request.runtime.context.get("env","test")
    if env == "prod":
        model = advanced_llm
        print("advanced_llm selected")
    else:
        model = basic_llm
        print("basic_llm selected")
    return handler(request.override(model=model))

In [58]:
agent2 = create_agent(
    model=basic_llm,
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
)

In [59]:
resp=agent2.invoke(
    input={"messages":[HumanMessage("C'est quoi un agent AI")]},
    context= {"env":"test"}
    )

[values] {'messages': [HumanMessage(content="C'est quoi un agent AI", additional_kwargs={}, response_metadata={}, id='15ea48e9-bd37-468c-b7ea-0b8bf80a2398')]}
basic_llm selected
[updates] {'model': {'messages': [AIMessage(content="Un agent AI (intelligence artificielle) est un système ou un programme informatique capable d'effectuer des tâches qui nécessitent normalement une intelligence humaine. Ces agents peuvent percevoir leur environnement, prendre des décisions, apprendre de nouvelles informations et interagir avec les utilisateurs ou d'autres systèmes.\n\nIl existe différents types d'agents AI, notamment :\n\n1. **Agents réactifs** : Ils réagissent à des stimuli de leur environnement sans mémoire ou capacité d'apprentissage. Par exemple, un programme qui joue aux échecs en suivant des règles prédéfinies.\n\n2. **Agents basés sur des modèles** : Ils utilisent des modèles internes pour représenter leur environnement et prendre des décisions basées sur ces modèles.\n\n3. **Agents d'

In [60]:
from IPython.display import Markdown

In [61]:
print(display(Markdown(resp['messages'][-1].content)))

Un agent AI (intelligence artificielle) est un système ou un programme informatique capable d'effectuer des tâches qui nécessitent normalement une intelligence humaine. Ces agents peuvent percevoir leur environnement, prendre des décisions, apprendre de nouvelles informations et interagir avec les utilisateurs ou d'autres systèmes.

Il existe différents types d'agents AI, notamment :

1. **Agents réactifs** : Ils réagissent à des stimuli de leur environnement sans mémoire ou capacité d'apprentissage. Par exemple, un programme qui joue aux échecs en suivant des règles prédéfinies.

2. **Agents basés sur des modèles** : Ils utilisent des modèles internes pour représenter leur environnement et prendre des décisions basées sur ces modèles.

3. **Agents d'apprentissage** : Ces agents peuvent apprendre de leurs expériences et améliorer leurs performances au fil du temps, comme les systèmes de recommandation ou les assistants virtuels.

4. **Agents autonomes** : Ils peuvent fonctionner de manière indépendante pour accomplir des tâches spécifiques, comme les drones ou les robots.

Les agents AI sont utilisés dans divers domaines, tels que la santé, la finance, le service client, les jeux vidéo, et bien d'autres, pour automatiser des processus, analyser des données et améliorer l'efficacité.

None


In [62]:
agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="You are a helpfull assistant"
)

In [63]:
resp=agent.invoke(input={"messages":[HumanMessage("Je m'appelle AYA")]})

In [64]:
print(resp['messages'][-1].content)

Bonjour Aya ! Comment puis-je vous aider aujourd'hui ?


In [65]:
resp=agent.invoke(input={"messages":[HumanMessage("C'est qoui mon prénom")]})

In [66]:
print(resp['messages'][-1].content)

Je suis désolé, mais je ne peux pas connaître ou deviner votre prénom. Toutefois, si vous avez besoin d'aide ou si vous souhaitez discuter d'un sujet en particulier, n'hésitez pas à me le faire savoir !


In [67]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()
agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="You are a helpfull assistant",
    checkpointer=memory
)

In [68]:
config = {"configurable":{"thread_id":1}}
resp=agent.invoke(
input={"messages":[HumanMessage("Je m'appelle Aya")]},
    config=config
    )

In [69]:
print(resp['messages'][-1].content)

Bonjour Aya ! Comment puis-je vous aider aujourd'hui ?


In [70]:
resp=agent.invoke(
input={"messages":[HumanMessage("Comment je m'appelle ?")]},
    config=config
    )

In [71]:
print(resp['messages'][-1].content)

Vous m'avez dit que vous vous appelez Aya.


In [72]:
from langchain.tools import tool 

In [73]:
@tool
def get_weather(city : str):
    """
    Get The weather of the given city
    """
    print("Weather Tool invoked")
    return{
        "city":city,
        "temperature":23,
        "humidity":80,
        "pressure":120
    }

@tool
def get_employee_info(employee_name : str):
    """
    Get infos about the given employee (salary, seniority)
    """
    print("get_employee_info Tool invoked")
    return{
        "name":employee_name,
        "salary":3400,
        "seniority":5
    }

In [74]:
agent4 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, get_employee_info],
    checkpointer=memory,
    system_prompt="answer the user question using only provided tools"
    )

In [75]:
config = {"configurable":{"thread_id":1}}
resp=agent4.invoke(input={'messages':[HumanMessage("La météo à Casablanca")]},config=config )
print(resp['messages'][-1].content)

Weather Tool invoked
La météo à Casablanca indique une température de 23°C avec une humidité de 80% et une pression atmosphérique de 120 hPa.


In [76]:
config = {"configurable":{"thread_id":1}}
resp=agent4.invoke(input={'messages':[HumanMessage("Quel est le salaire de Hassan?")]},config=config )
print(resp['messages'][-1].content)

get_employee_info Tool invoked
Le salaire de Hassan est de 3400, et il a une ancienneté de 5 ans.


In [77]:
load_dotenv(override=True)

True

In [80]:
from langchain_tavily import TavilySearch

In [81]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

In [86]:
@tool
def search_web(query:str):
    """
    Search for general current web results
    """
    print(f"search_web invoked with{query}")
    results = tavily.invoke({"query":query})
    return results

In [87]:
agent5 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, get_employee_info, search_web],
    checkpointer=memory,
    system_prompt="answer the user question using only provided tools"
    )

In [92]:
resp=agent5.invoke(
input={"messages":[HumanMessage("Actualités sur le Maroc")]},
    config=config
    )

search_web invoked withactualités sur le Maroc


In [93]:
print(display(Markdown(resp['messages'][-1].content)))

Voici quelques actualités récentes concernant le Maroc :

1. **Inondations à Safi** : Au moins 37 personnes sont mortes au Maroc après des inondations soudaines qui ont touché la région de Safi, située à 300 kilomètres de Rabat. ([source](https://www.franceinfo.fr/monde/afrique/maroc/))

2. **Classement du bonheur** : Le Maroc est maintenu à la 112e place du classement du bonheur en 2026, selon un rapport mondial sur le bien-être. ([source](https://lematin.ma/tag/maroc-2026))

3. **Relations internationales** : Le Maroc continue de développer ses relations internationales, notamment en renforçant sa coopération militaire avec la France. D'importants accords de partenariat ont aussi été signés avec la République Tchèque sur le Sahara. ([source](https://www.jeuneafrique.com/pays/maroc/))

4. **Tourisme** : Le Maroc a accueilli près de 20 millions de visiteurs en 2025, et la dynamique devrait se poursuivre en 2026. ([source](https://fr.hespress.com/463073-maroc-croissance-confirmee-en-2026-grands-chantiers-et-services-en-locomotive.html))

Pour plus d'informations, vous pouvez consulter des portails comme Franceinfo, Le Matin, et Jeune Afrique.

None


In [97]:
from langchain_experimental.tools import PythonREPLTool

In [109]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [114]:
agent6 = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="""Générer et exécuter le code python en plaçant le code python généré 
    et le résultat d'exécution de ce code dans le fichier doc.txt""",
    debug=True
    )

In [115]:
resp = agent6.invoke(input={"messages":[
    HumanMessage("Je veux trier les deux listes [6,3,7,4] et [6,8,3,2] et puis aditionner les deux listes")
]})

[values] {'messages': [HumanMessage(content='Je veux trier les deux listes [6,3,7,4] et [6,8,3,2] et puis aditionner les deux listes', additional_kwargs={}, response_metadata={}, id='ad9eec7a-36af-4d40-bbf4-48381ab503ad')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 147, 'total_tokens': 253, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_899261a2ad', 'id': 'chatcmpl-DPW3h6O4gCQ3MdRFV1MdCnpmZTswM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d44ae-e528-77a1-8e3b-9b2ce09b5e65-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'liste1 = [6,3,7,4]\\nliste2 = [6,8,3,2

In [116]:
 print(resp['messages'][-1].content)

Le code et les résultats ont été enregistrés avec succès dans le fichier `doc.txt`.
